In [1]:
import boto3

s3 = boto3.client("s3", endpoint_url="http://minio:9000",
                  aws_access_key_id="minioadmin", aws_secret_access_key="minioadmin")
s3.upload_file("../data/orders_history.jsonl", "lake", "bronze/history/orders_history.jsonl")
s3.upload_file("../data/restaurants.json", "lake", "bronze/referentiel/restaurants.json")
[o["Key"] for o in s3.list_objects_v2(Bucket="lake")["Contents"]]

['bronze/history/orders_history.jsonl', 'bronze/referentiel/restaurants.json']

In [2]:
import json, time
from operator import add
from spark_session import get_spark

spark = get_spark("FastFood-P2-RDD")
sc = spark.sparkContext

https://repos.spark-packages.org added as a remote repository with the name: repo-1


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0146f4e6-aa14-4340-a961-cc241808f1de;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
downloading https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar ...
	[SUCCESSFUL ] org.apache.hadoop#hadoop-aws;3.3.4!hadoop-aws.jar (274ms)
downloading https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar ...
	[SUCCESSFUL ] com.amazonaws#aws-java-sdk-bundle;1.12.262!aws-java-sdk-bundle.jar (27176ms)
downloading https://repo1.maven.org/maven2/org/wildfly/openssl/wildfly-openssl/1.0.7.Final/wildfly-

In [3]:
t0 = time.time()

raw = sc.textFile("s3a://lake/bronze/history/orders_history.jsonl")      # transformation
events = raw.map(json.loads)                                              # transformation

# Dédoublonnage (~2 % de doublons order_id + status) pour ne pas compter 2 fois le CA
dedup = (events.map(lambda e: ((e["order_id"], e["status"]), e))          # transformation
               .reduceByKey(lambda a, b: a)                               # transformation (shuffle)
               .values())

delivered = dedup.filter(lambda e: e["status"] == "DELIVERED").cache()   # transformation
print("Événements bruts  :", events.count())                             # action
print("Commandes livrées :", delivered.count())                          # action

26/11/13 22:22:59 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

Événements bruts  : 50003


[Stage 2:>                                                          (0 + 2) / 2]

Commandes livrées : 14968


In [4]:
ca_ville = (delivered.map(lambda e: (e["city"], e["amount_mad"]))
                     .reduceByKey(add))

# somme et nombre en un seul passage
panier = (delivered.map(lambda e: (e["city"], e["amount_mad"]))
                   .aggregateByKey((0.0, 0),
                                   lambda acc, v: (acc[0] + v, acc[1] + 1),       # dans une partition
                                   lambda a, b: (a[0] + b[0], a[1] + b[1]))       # entre partitions
                   .mapValues(lambda s: (round(s[0], 2), s[1], round(s[0] / s[1], 2))))

print(f"{'Ville':<12}{'CA (MAD)':>15}{'Nb':>8}{'Panier moyen':>15}")
for ville, (ca, n, moy) in sorted(panier.collect(), key=lambda x: -x[1][0]):    # action
    print(f"{ville:<12}{ca:>15,.2f}{n:>8}{moy:>15.2f}")

Ville              CA (MAD)      Nb   Panier moyen
Casablanca     1,405,702.07    6010         233.89
Rabat            832,840.93    3726         223.52
Marrakech        673,953.99    3026         222.72
Tanger           493,889.59    2206         223.88


In [6]:
top5 = (delivered.map(lambda e: (e["restaurant_id"], 1))
                 .reduceByKey(add)
                 .takeOrdered(5, key=lambda x: -x[1]))                    # action
for r, n in top5:
    print(r, n)

t_rdd = time.time() - t0
print(f"\nTemps total Partie 2 (RDD) : {t_rdd:.2f} s")

R-199 907
R-097 416
R-087 412
R-151 341
R-061 223

Temps total Partie 2 (RDD) : 103.17 s


In [7]:
print(ca_ville.toDebugString().decode())

(2) PythonRDD[23] at RDD at PythonRDD.scala:53 []
 |  MapPartitionsRDD[12] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[11] at partitionBy at <unknown>:0 []
 +-(2) PairwiseRDD[10] at reduceByKey at /tmp/ipykernel_344/243674633.py:1 []
    |  PythonRDD[9] at reduceByKey at /tmp/ipykernel_344/243674633.py:1 []
    |  PythonRDD[6] at RDD at PythonRDD.scala:53 []
    |      CachedPartitions: 2; MemorySize: 755.6 KiB; DiskSize: 0.0 B
    |  MapPartitionsRDD[5] at mapPartitions at PythonRDD.scala:160 []
    |  ShuffledRDD[4] at partitionBy at <unknown>:0 []
    +-(2) PairwiseRDD[3] at reduceByKey at /tmp/ipykernel_344/2041855568.py:7 []
       |  PythonRDD[2] at reduceByKey at /tmp/ipykernel_344/2041855568.py:7 []
       |  s3a://lake/bronze/history/orders_history.jsonl MapPartitionsRDD[1] at textFile at <unknown>:0 []
       |  s3a://lake/bronze/history/orders_history.jsonl HadoopRDD[0] at textFile at <unknown>:0 []


In [8]:
def ca_par_ville_rdd():
    return (sc.textFile("s3a://lake/bronze/history/orders_history.jsonl")
              .map(json.loads)
              .map(lambda e: ((e["order_id"], e["status"]), e))
              .reduceByKey(lambda a, b: a)
              .values()
              .filter(lambda e: e["status"] == "DELIVERED")
              .map(lambda e: (e["city"], e["amount_mad"]))
              .reduceByKey(add)
              .collect())

ca_par_ville_rdd()                      # 1er run : chauffe (JVM, connexions S3)
temps = []
for _ in range(3):
    t = time.time(); ca_par_ville_rdd(); temps.append(time.time() - t)
print("Temps RDD (s) :", [round(x, 2) for x in temps], "→ médiane", round(sorted(temps)[1], 2))

[Stage 19:>                                                         (0 + 2) / 2]

Temps RDD (s) : [3.08, 2.07, 2.15] → médiane 2.15


## Partie 2 – Analyse

### Transformations (lazy) et actions
- **Transformations** : `textFile`, `map`, `filter`, `reduceByKey`, `aggregateByKey`, `mapValues`, `values`, `cache`. Elles ne calculent rien : elles ajoutent un nœud au DAG.
- **Actions** : `count`, `collect`, `takeOrdered`. Elles déclenchent un job, que le driver découpe en stages puis en tasks (une task par partition, ici 2 partitions).

### Lineage, stages et shuffle
Le `toDebugString()` de `ca_ville` montre deux indentations `+-`, soit **deux frontières de shuffle** :
1. `textFile → map(json.loads) → map(clé)` puis le shuffle de dédoublonnage (`reduceByKey` sur `(order_id, status)`) ;
2. `values → filter → cache → map(ville, montant)` puis le shuffle du `reduceByKey` sur la ville ;
3. un stage final qui produit le résultat.

Le job `collect` compte donc **3 stages**. Le premier apparaît « skipped » dans Spark UI, car `delivered` est en cache (`CachedPartitions: 2`).

**Le shuffle de `reduceByKey`** : toutes les valeurs d'une même clé (ville) doivent arriver dans la même partition. Chaque executor fait d'abord une **pré-agrégation locale** (combiner, côté map), écrit ses résultats partiels sur disque local, puis les executors du stage suivant les récupèrent par le réseau. C'est l'opération la plus coûteuse (sérialisation, disque, réseau), et c'est elle qui coupe le DAG en stages. `reduceByKey` est préférable à `groupByKey`, car la pré-agrégation réduit fortement le volume échangé (4 clés seulement ici).

### Tolérance aux pannes par le lineage
Un RDD est immuable et connaît la suite de transformations qui l'a produit à partir d'une source fiable (MinIO). Si un executor tombe et qu'une partition est perdue, Spark **ne réplique pas les données** : il **recalcule uniquement la partition perdue** en rejouant son lineage depuis la source, ou depuis la dernière sortie de shuffle encore disponible. Le `cache()` accélère les réutilisations, mais n'est pas nécessaire à la reprise : une partition en cache perdue est elle aussi recalculée.

### Résultats
Mettre ici le tableau CA / panier moyen par ville, le top 5 (R-199 : 907, R-097 : 416, R-087 : 412, R-151 : 341, R-061 : 223) et la capture du DAG.

In [9]:
spark.stop()